# Automotive Part Image-Text Matching

## Development experiment

This notebook presents the current development experiment in a compact form.

The task has three classes:

- **MATCH** — the image and description refer to the same part category.
- **PARTIAL_MATCH** — the description is from the same automotive system, but it is a different part category.
- **MISMATCH** — the description is from a different automotive system.

The notebook uses the committed development data and validation reports. It does **not** load or evaluate the test split.


## 1. Experiment plan

The comparison contains:

1. Majority baseline
2. TF-IDF + Logistic Regression
3. Image pixels + Logistic Regression
4. Keras text neural network
5. Keras image neural network
6. Keras multimodal neural network

The main question is whether combining the image and the text works better than using only one modality.


In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, Markdown


def find_repository_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "README.md").is_file() and (candidate / "src").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter from the project folder "
        "or keep this notebook inside the notebooks directory."
    )


ROOT = find_repository_root()
print(f"Repository root: {ROOT}")

for package_name in ["python", "numpy", "pandas", "matplotlib", "pillow", "tensorflow"]:
    if package_name == "python":
        version = sys.version.split()[0]
    else:
        try:
            version = metadata.version(package_name)
        except metadata.PackageNotFoundError:
            version = "not installed"
    print(f"{package_name:12s}: {version}")


## 2. Load the development data

Only the development metadata, train split, validation split, and split manifest are loaded below.

The test CSV remains unopened because the final model selection procedure has not been fixed yet.


In [ ]:
METADATA_CSV = ROOT / "data" / "development" / "metadata.csv"
TRAIN_CSV = ROOT / "data" / "processed" / "development_train.csv"
VALIDATION_CSV = ROOT / "data" / "processed" / "development_validation.csv"
SPLIT_MANIFEST_CSV = ROOT / "data" / "processed" / "development_split_manifest.csv"
TEST_CSV = ROOT / "data" / "processed" / "development_test.csv"  # path only; not read

required_files = [
    METADATA_CSV,
    TRAIN_CSV,
    VALIDATION_CSV,
    SPLIT_MANIFEST_CSV,
]

missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    formatted = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(f"Required project files are missing:\n{formatted}")

metadata_df = pd.read_csv(METADATA_CSV)
train_df = pd.read_csv(TRAIN_CSV)
validation_df = pd.read_csv(VALIDATION_CSV)
split_manifest_df = pd.read_csv(SPLIT_MANIFEST_CSV)

print(f"Development samples: {len(metadata_df)}")
print(f"Train samples:       {len(train_df)}")
print(f"Validation samples:  {len(validation_df)}")
print(f"Test file exists:    {TEST_CSV.is_file()} (not loaded)")


In [ ]:
summary_df = pd.DataFrame(
    {
        "samples": [
            len(metadata_df),
            len(train_df),
            len(validation_df),
        ],
        "physical_part_groups": [
            metadata_df["part_group_id"].nunique(),
            train_df["part_group_id"].nunique(),
            validation_df["part_group_id"].nunique(),
        ],
        "unique_images": [
            metadata_df["image_id"].nunique(),
            train_df["image_id"].nunique(),
            validation_df["image_id"].nunique(),
        ],
    },
    index=["development", "train", "validation"],
)

display(summary_df)


## 3. Class and category balance

In [ ]:
label_balance = (
    pd.concat(
        [
            train_df["label"].value_counts().rename("train"),
            validation_df["label"].value_counts().rename("validation"),
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
)

display(label_balance)

ax = label_balance.plot(kind="bar", figsize=(8, 4))
ax.set_title("Label distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Samples")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
category_balance = (
    metadata_df.groupby(["part_family", "part_category"])["part_group_id"]
    .nunique()
    .rename("physical_part_groups")
    .reset_index()
    .sort_values(["part_family", "part_category"])
)

display(category_balance)


## 4. Group split integrity

Rows from one physical part must remain in a single split. This prevents the same part from appearing in both training and validation data.


In [ ]:
train_groups = set(train_df["part_group_id"])
validation_groups = set(validation_df["part_group_id"])

train_validation_overlap = train_groups & validation_groups

if train_validation_overlap:
    raise AssertionError(
        f"Group leakage detected between train and validation: "
        f"{sorted(train_validation_overlap)}"
    )

manifest_group_rows = split_manifest_df.groupby("part_group_id").size()
duplicate_manifest_groups = manifest_group_rows[manifest_group_rows > 1]

if not duplicate_manifest_groups.empty:
    raise AssertionError(
        "A physical part group appears more than once in the split manifest."
    )

print("Train/validation group overlap: 0")
print("Split manifest duplicate groups: 0")

manifest_split_column = next(
    (column for column in ["split", "dataset_split", "assigned_split"] if column in split_manifest_df.columns),
    None,
)

if manifest_split_column:
    display(
        split_manifest_df[manifest_split_column]
        .value_counts()
        .rename("physical_part_groups")
        .to_frame()
    )
else:
    print("Split counts are available in the manifest, but the split column has a different name.")


## 5. One image with three descriptions

The development data intentionally pairs the same image with three descriptions. This explains why the image-only models cannot identify the relationship class reliably.


In [ ]:
example_image_id = metadata_df["image_id"].iloc[0]
example_rows = (
    metadata_df.loc[metadata_df["image_id"] == example_image_id]
    .sort_values("label")
    .reset_index(drop=True)
)

image_relative_path = Path(example_rows.loc[0, "image_path"])
image_path = ROOT / image_relative_path

if image_path.is_file():
    image = Image.open(image_path).convert("RGB")
    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.title(
        f"{example_rows.loc[0, 'part_category']} | "
        f"{example_rows.loc[0, 'part_group_id']}"
    )
    plt.axis("off")
    plt.show()
else:
    print(f"Image file not found: {image_path}")

display(example_rows[["description", "label", "part_family", "part_category"]])


## 6. Optional experiment rebuild

By default this notebook reads the committed experiment artifacts and does not retrain the models.

Set `RUN_TRAINING = True` only when a complete rebuild is required. Neural training can take time and small numeric differences are possible across TensorFlow builds and hardware.


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    commands = [
        "create-development-data",
        "validate-development-data",
        "create-grouped-split",
        "run-baselines",
        "train-text",
        "train-image",
        "train-multimodal",
    ]

    for command in commands:
        print(f"Running: {command}")
        subprocess.run(
            [sys.executable, "-m", "src.project_cli", command],
            cwd=ROOT,
            check=True,
        )
else:
    print("Training is disabled. Using committed validation reports.")


## 7. Validation results

In [ ]:
METRIC_FILES = [
    ROOT / "reports" / "baselines" / "majority_validation_metrics.json",
    ROOT / "reports" / "baselines" / "text_tfidf_logistic_regression_validation_metrics.json",
    ROOT / "reports" / "baselines" / "image_pixels_logistic_regression_validation_metrics.json",
    ROOT / "reports" / "keras_text" / "validation_metrics.json",
    ROOT / "reports" / "keras_image" / "validation_metrics.json",
    ROOT / "reports" / "keras_multimodal" / "validation_metrics.json",
]

metric_records = []
metrics_by_model = {}

for path in METRIC_FILES:
    if not path.is_file():
        raise FileNotFoundError(f"Validation metrics file not found: {path}")

    with path.open("r", encoding="utf-8-sig") as file:
        payload = json.load(file)

    model_name = payload["model"]
    metrics_by_model[model_name] = payload
    metric_records.append(
        {
            "model": model_name,
            "accuracy": float(payload["accuracy"]),
            "macro_f1": float(payload["macro_f1"]),
            "validation_samples": int(payload["sample_count"]),
            "test_split_used": payload.get("test_split_used", False),
        }
    )

results_df = (
    pd.DataFrame(metric_records)
    .sort_values(["macro_f1", "accuracy"], ascending=False)
    .reset_index(drop=True)
)

display(
    results_df.style.format(
        {
            "accuracy": "{:.4f}",
            "macro_f1": "{:.4f}",
        }
    )
)


In [ ]:
plot_df = results_df.set_index("model")[["accuracy", "macro_f1"]].sort_values("macro_f1")

ax = plot_df.plot(kind="barh", figsize=(10, 6))
ax.set_title("Validation model comparison")
ax.set_xlabel("Score")
ax.set_ylabel("")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()


In [ ]:
best_row = results_df.iloc[0]

display(
    Markdown(
        f"**Best validation model:** {best_row['model']}  \n"
        f"Accuracy: **{best_row['accuracy']:.4f}**  \n"
        f"Macro F1: **{best_row['macro_f1']:.4f}**"
    )
)


## 8. Best model confusion matrix

In [ ]:
best_payload = metrics_by_model[best_row["model"]]
confusion_matrix = np.asarray(best_payload["confusion_matrix"])
confusion_labels = best_payload["confusion_matrix_labels"]

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(confusion_matrix)
ax.set_title(f"{best_row['model']} — validation confusion matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_xticks(range(len(confusion_labels)), confusion_labels, rotation=30, ha="right")
ax.set_yticks(range(len(confusion_labels)), confusion_labels)

for row_index in range(confusion_matrix.shape[0]):
    for column_index in range(confusion_matrix.shape[1]):
        ax.text(
            column_index,
            row_index,
            str(confusion_matrix[row_index, column_index]),
            ha="center",
            va="center",
        )

fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
per_class_df = (
    pd.DataFrame(best_payload["per_class"])
    .T
    .loc[:, ["precision", "recall", "f1", "support"]]
)

display(
    per_class_df.style.format(
        {
            "precision": "{:.4f}",
            "recall": "{:.4f}",
            "f1": "{:.4f}",
            "support": "{:.0f}",
        }
    )
)


## 9. Multimodal prediction examples

In [ ]:
MULTIMODAL_PREDICTIONS_CSV = (
    ROOT / "reports" / "keras_multimodal" / "validation_predictions.csv"
)

multimodal_predictions_df = pd.read_csv(MULTIMODAL_PREDICTIONS_CSV)

prediction_columns = [
    "sample_id",
    "part_category",
    "description",
    "true_label",
    "predicted_label",
    "is_correct",
    "probability_MATCH",
    "probability_PARTIAL_MATCH",
    "probability_MISMATCH",
]

display(multimodal_predictions_df[prediction_columns].head(10))


In [ ]:
correct_predictions = multimodal_predictions_df[
    multimodal_predictions_df["is_correct"].astype(str).str.lower().isin(["true", "1"])
]

incorrect_predictions = multimodal_predictions_df[
    ~multimodal_predictions_df.index.isin(correct_predictions.index)
]

print(f"Correct predictions:   {len(correct_predictions)}")
print(f"Incorrect predictions: {len(incorrect_predictions)}")

if not incorrect_predictions.empty:
    display(
        incorrect_predictions[
            [
                "sample_id",
                "part_category",
                "description",
                "true_label",
                "predicted_label",
                "probability_MATCH",
                "probability_PARTIAL_MATCH",
                "probability_MISMATCH",
            ]
        ].reset_index(drop=True)
    )


## 10. Neural training history

In [ ]:
HISTORY_FILES = {
    "Keras text": ROOT / "reports" / "keras_text" / "training_history.csv",
    "Keras image": ROOT / "reports" / "keras_image" / "training_history.csv",
    "Keras multimodal": ROOT / "reports" / "keras_multimodal" / "training_history.csv",
}

for model_name, history_path in HISTORY_FILES.items():
    if not history_path.is_file():
        print(f"Missing history file: {history_path}")
        continue

    history_df = pd.read_csv(history_path)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history_df["epoch"], history_df["accuracy"], label="train accuracy")
    ax.plot(history_df["epoch"], history_df["val_accuracy"], label="validation accuracy")
    ax.set_title(f"{model_name} — accuracy")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.legend()
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history_df["epoch"], history_df["loss"], label="train loss")
    ax.plot(history_df["epoch"], history_df["val_loss"], label="validation loss")
    ax.set_title(f"{model_name} — loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    plt.tight_layout()
    plt.show()


## 11. Multimodal model architecture

In [ ]:
ARCHITECTURE_FILE = (
    ROOT / "reports" / "keras_multimodal" / "model_architecture.txt"
)

if ARCHITECTURE_FILE.is_file():
    architecture_text = ARCHITECTURE_FILE.read_text(encoding="utf-8-sig")
    print(architecture_text)
else:
    print(f"Architecture report not found: {ARCHITECTURE_FILE}")


## 12. Conclusions

- The majority and image-only models remain near the three-class baseline.
- The text-only models use useful information from the description, but they do not solve the complete matching task.
- The multimodal model gives the best current validation result because it can compare information from both inputs.
- The current dataset is generated and small, so these results are development results only.
- The test split remains locked and was not used in this notebook.
- The next important experiment is evaluation with approved real automotive-part photographs.


## 13. Reproducibility commands

Run these commands from the repository root when a complete rebuild is needed:

```powershell
python -m src.project_cli create-development-data
python -m src.project_cli validate-development-data
python -m src.project_cli create-grouped-split
python -m src.project_cli run-baselines
python -m src.project_cli train-text
python -m src.project_cli train-image
python -m src.project_cli train-multimodal
python -m pytest -q
python -m src.project_cli verify-development-pipeline
```

Open this notebook with:

```powershell
python -m jupyter notebook notebooks/01_development_experiment.ipynb
```
